# Appendix A3: Modern Architectures for Cognitive Modeling
## From Vanilla RNNs to State Space Models and Beyond

**Spinning Up in Active Inference** | Appendix

---

The landscape of neural network architectures for cognitive modeling has exploded beyond vanilla RNNs. This notebook surveys the zoo of modern architectures — continuous-time RNNs, state space models, attention mechanisms, and tiny interpretable networks — with hands-on NumPy implementations that make each architecture's computational principles transparent.

Every implementation here runs in pure NumPy. We sacrifice GPU speed for interpretability: you can inspect every weight, trace every gradient, and plot every dynamical trajectory.

By the end of this notebook you will:
1. Implement a vanilla RNN, CTRNN, and gated RNN from scratch
2. Understand how continuous-time dynamics differ from discrete updates
3. See how attention mechanisms implement input gating
4. Analyze network dynamics using phase portraits and fixed-point analysis
5. Connect each architecture to Active Inference concepts

**Prerequisites:** Module 7 (Deep RL), linear algebra basics.  
**Time:** ~90 minutes.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy.integrate import solve_ivp
import sys; sys.path.insert(0, '..')
from utils.plotting import COLORS, _setup_style, dual_perspective_box, plot_matrix_heatmap
_setup_style()

np.random.seed(42)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def tanh(x):
    return np.tanh(x)

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

print("Setup complete — all implementations in pure NumPy.")

## A3.1 Architecture Zoo Overview

The table below summarizes the key architectures we will implement and analyze. Each sits at a different point in the trade-off space between biological plausibility, computational efficiency, and interpretability.

| Architecture | Update Rule | Time | Key Property | Cognitive Relevance |
|---|---|---|---|---|
| **Vanilla RNN** | $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$ | Discrete | Simple, interpretable | Baseline |
| **CTRNN** | $\tau \frac{dh}{dt} = -h + \tanh(Wh + Ux)$ | Continuous | Biological timescales | Neural dynamics |
| **GRU/LSTM** | Gated updates | Discrete | Long-range memory | Working memory |
| **Attention** | Scaled dot-product | Discrete | Content-based selection | Input gating |
| **SSM (S4/Mamba)** | $\dot{x} = Ax + Bu$ | Continuous | Linear recurrence | Kalman filter connection |
| **Tiny RNN (1–4 units)** | Standard RNN | Discrete | Full interpretability | Phase portrait analysis |

A recurring theme: **gating** and **sustained activity** keep appearing across architectures. Whether it is the forget gate in an LSTM, the attention mask in a Transformer, or the time constant in a CTRNN, every architecture must solve the same fundamental problem: *what to remember and what to forget*.

From an Active Inference perspective, this is precision-weighting: the agent must modulate the influence of incoming sensory data vs. prior beliefs.

## A3.2 Vanilla RNN Baseline

We start with the simplest recurrent architecture: the vanilla RNN. The update rule is:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b_h)$$
$$y_t = W_y h_t + b_y$$

To test it, we use a **3-bit flip-flop** task: input pulses toggle stored bits ($+1$ or $-1$), and the network must maintain the current state of each bit between pulses. This is a minimal working memory benchmark — the network must sustain activity in the absence of input.

In [ ]:
# ── Toy Working Memory Task ───────────────────────────────────────────────────────
def generate_flip_flop_data(n_trials=100, T=50, n_bits=3):
    """3-bit flip-flop: input pulses toggle stored bits."""
    inputs = np.zeros((n_trials, T, n_bits))
    targets = np.zeros((n_trials, T, n_bits))

    for trial in range(n_trials):
        state = np.zeros(n_bits)
        for t in range(T):
            if np.random.random() < 0.1:  # 10% chance of input pulse
                bit = np.random.randint(n_bits)
                val = np.random.choice([-1, 1])
                inputs[trial, t, bit] = val
                state[bit] = val
            targets[trial, t] = state

    return inputs, targets

inputs, targets = generate_flip_flop_data()
print(f"Data shape: inputs {inputs.shape}, targets {targets.shape}")
print(f"Sparsity: {(inputs != 0).mean():.3f} of input entries are nonzero")

In [ ]:
# ── Vanilla RNN Implementation ─────────────────────────────────────────────────
class VanillaRNN:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        scale = 0.1
        self.Wh = np.random.randn(hidden_size, hidden_size) * scale
        self.Wx = np.random.randn(hidden_size, input_size) * scale
        self.Wy = np.random.randn(output_size, hidden_size) * scale
        self.bh = np.zeros(hidden_size)
        self.by = np.zeros(output_size)
        self.hidden_size = hidden_size
        self.lr = lr

    def forward(self, x_seq, h0=None):
        T = len(x_seq)
        if h0 is None:
            h0 = np.zeros(self.hidden_size)

        h = h0
        hiddens, outputs = [], []
        for t in range(T):
            h = np.tanh(self.Wh @ h + self.Wx @ x_seq[t] + self.bh)
            y = self.Wy @ h + self.by
            hiddens.append(h.copy())
            outputs.append(y.copy())

        return np.array(hiddens), np.array(outputs)

# Create and run
rnn = VanillaRNN(input_size=3, hidden_size=16, output_size=3)
trial_idx = 0
hiddens, outputs = rnn.forward(inputs[trial_idx])
print(f"Hidden states shape: {hiddens.shape}")
print(f"Outputs shape: {outputs.shape}")

In [ ]:
# ── Visualize Vanilla RNN on Flip-Flop Task ─────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

trial_idx = 0
T = inputs.shape[1]
time = np.arange(T)

# Input pulses
for bit in range(3):
    axes[0].stem(time, inputs[trial_idx, :, bit], linefmt=f'C{bit}-',
                 markerfmt=f'C{bit}o', basefmt='k-', label=f'Bit {bit}')
axes[0].set_ylabel('Input pulses')
axes[0].set_title('3-Bit Flip-Flop Task', fontsize=14)
axes[0].legend(loc='upper right', fontsize=9)

# Target states
for bit in range(3):
    axes[1].plot(time, targets[trial_idx, :, bit], f'C{bit}-', linewidth=2, label=f'Bit {bit}')
axes[1].set_ylabel('Target state')
axes[1].set_title('Target (what the network should maintain)', fontsize=11)

# RNN output (untrained)
for bit in range(3):
    axes[2].plot(time, outputs[:, bit], f'C{bit}--', linewidth=1.5, alpha=0.8, label=f'Bit {bit}')
axes[2].set_ylabel('RNN output')
axes[2].set_title('Untrained RNN output (random weights)', fontsize=11)

# Hidden state trajectories (first 6 units)
for unit in range(min(6, hiddens.shape[1])):
    axes[3].plot(time, hiddens[:, unit], linewidth=0.8, alpha=0.7)
axes[3].set_ylabel('Hidden units')
axes[3].set_xlabel('Time step')
axes[3].set_title('Hidden state trajectories (first 6 units)', fontsize=11)

plt.tight_layout()
plt.show()

print("Note: The untrained RNN cannot sustain states between pulses.")
print("Training it to do so requires learning attractors in hidden-state space.")

## A3.3 Continuous-Time RNN (CTRNN)

Biological neurons do not update in lockstep discrete time steps. The CTRNN replaces the discrete update with a continuous-time ODE:

$$\tau_i \frac{dh_i}{dt} = -h_i + \tanh\left(\sum_j W_{ij} h_j + \sum_k U_{ik} x_k\right)$$

The time constant $\tau_i$ controls how quickly each unit responds. This is biologically critical: different brain regions operate on different timescales (sensory cortex: ~10ms, prefrontal cortex: ~100ms+). The CTRNN naturally captures this heterogeneity.

**Key insight:** The $-h$ term provides a natural decay — without input, activity fades. Sustained activity (working memory) requires the recurrent connections $W$ to create **attractors** that counteract this decay.

In [ ]:
# ── CTRNN Implementation ───────────────────────────────────────────────────────
class CTRNN:
    def __init__(self, n_units, tau=None, dt=0.1):
        self.n = n_units
        self.W = np.random.randn(n_units, n_units) * 0.5 / np.sqrt(n_units)
        if tau is None:
            tau = np.ones(n_units) * 10.0  # default 10 time units
        self.tau = tau
        self.dt = dt

    def step(self, h, x=None):
        """Euler integration step."""
        if x is None:
            x = np.zeros(self.n)
        dhdt = (-h + np.tanh(self.W @ h + x)) / self.tau
        return h + self.dt * dhdt

    def run(self, h0, x_seq=None, T=100):
        h = h0.copy()
        trajectory = [h.copy()]
        for t in range(T):
            x = x_seq[t] if x_seq is not None and t < len(x_seq) else None
            h = self.step(h, x)
            trajectory.append(h.copy())
        return np.array(trajectory)

# Demonstrate with different tau values
np.random.seed(42)
n_units = 5
W_shared = np.random.randn(n_units, n_units) * 0.5 / np.sqrt(n_units)
h0 = np.random.randn(n_units) * 0.5

# Create a transient input pulse
T = 300
x_seq = np.zeros((T, n_units))
x_seq[20:30] = np.random.randn(10, n_units) * 2.0  # pulse at t=20-30

taus = [1.0, 5.0, 20.0]
trajectories = {}
for tau_val in taus:
    net = CTRNN(n_units, tau=np.ones(n_units) * tau_val, dt=0.1)
    net.W = W_shared.copy()
    trajectories[tau_val] = net.run(h0.copy(), x_seq=x_seq, T=T)

print(f"Trajectory shapes: {trajectories[1.0].shape}")

In [ ]:
# ── Effect of Tau on CTRNN Dynamics ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for idx, tau_val in enumerate(taus):
    traj = trajectories[tau_val]
    t_axis = np.arange(traj.shape[0]) * 0.1  # convert to continuous time
    for unit in range(n_units):
        axes[idx].plot(t_axis, traj[:, unit], linewidth=1.2, alpha=0.8)
    axes[idx].axvspan(2.0, 3.0, alpha=0.15, color=COLORS['highlight'], label='Input pulse')
    axes[idx].set_title(f'$\\tau = {tau_val}$', fontsize=13)
    axes[idx].set_xlabel('Time')
    if idx == 0:
        axes[idx].set_ylabel('Unit activation')
    axes[idx].legend(fontsize=8, loc='upper right')

fig.suptitle('CTRNN Dynamics: Time Constants Control Memory Duration',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Fast tau (1.0): Activity decays quickly after the pulse — no working memory.")
print("Medium tau (5.0): Sustained activity for several time units — short-term memory.")
print("Slow tau (20.0): Very slow dynamics — the network integrates input over long windows.")

In [ ]:
# ── CTRNN with scipy.integrate for Higher Accuracy ─────────────────────────
# Euler integration can be inaccurate for stiff systems.
# Here we use solve_ivp for a 2-unit CTRNN to get ground-truth dynamics.

np.random.seed(123)
W2 = np.array([[1.5, -0.8],
               [0.6,  1.2]])  # Hand-designed for interesting dynamics
tau2 = np.array([5.0, 10.0])

def ctrnn_ode(t, h, W, tau, x_func):
    x = x_func(t)
    dhdt = (-h + np.tanh(W @ h + x)) / tau
    return dhdt

def x_pulse(t):
    """Input pulse between t=2 and t=3."""
    if 2.0 <= t <= 3.0:
        return np.array([1.0, 0.0])
    return np.array([0.0, 0.0])

h0_2d = np.array([0.0, 0.0])
t_span = (0, 30)
t_eval = np.linspace(0, 30, 500)

sol = solve_ivp(ctrnn_ode, t_span, h0_2d, t_eval=t_eval,
                args=(W2, tau2, x_pulse), method='RK45')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time series
ax1.plot(sol.t, sol.y[0], color=COLORS['rl'], linewidth=2, label=f'$h_1$ ($\\tau$={tau2[0]})')
ax1.plot(sol.t, sol.y[1], color=COLORS['aif'], linewidth=2, label=f'$h_2$ ($\\tau$={tau2[1]})')
ax1.axvspan(2.0, 3.0, alpha=0.15, color=COLORS['highlight'], label='Input pulse')
ax1.set_xlabel('Time')
ax1.set_ylabel('Activation')
ax1.set_title('2-Unit CTRNN: Heterogeneous Time Constants', fontsize=13)
ax1.legend()

# Phase portrait (h1 vs h2)
ax2.plot(sol.y[0], sol.y[1], color=COLORS['neutral'], linewidth=1.5, alpha=0.7)
ax2.plot(sol.y[0, 0], sol.y[1, 0], 'o', color=COLORS['reward'], markersize=10, label='Start')
ax2.plot(sol.y[0, -1], sol.y[1, -1], 's', color=COLORS['highlight'], markersize=10, label='End')
ax2.set_xlabel('$h_1$')
ax2.set_ylabel('$h_2$')
ax2.set_title('Phase Space Trajectory', fontsize=13)
ax2.legend()

plt.tight_layout()
plt.show()

print("Different tau values cause h1 to respond fast and h2 to respond slow.")
print("This temporal hierarchy is how the brain separates fast sensory from slow cognitive processing.")

In [ ]:
# ── Dual Perspective: Discrete vs Continuous Time ──────────────────────────
dual_perspective_box(
    rl_text=(
        "Standard deep RL uses discrete timesteps: one observation, one action, "
        "one reward per tick. RNNs in RL (e.g., R2D2, DreamerV3) inherit this "
        "discrete structure. The choice of tick rate is a hyperparameter — "
        "too slow and you miss fast dynamics, too fast and training is expensive."
    ),
    aif_text=(
        "Active Inference is naturally formulated in continuous time via "
        "generalized coordinates. The CTRNN's tau parameters map onto the "
        "hierarchical timescales in predictive coding: fast prediction errors "
        "at low levels, slow contextual beliefs at high levels. Precision-weighting "
        "modulates effective time constants."
    ),
    title="Discrete Time (RL) vs. Continuous Time (Active Inference)"
)

## A3.4 Gating and Attention

### The Gating Principle

Vanilla RNNs struggle with long-range dependencies because gradients vanish or explode through the $\tanh$ nonlinearity. **Gating** solves this by introducing multiplicative interactions that can selectively pass or block information flow.

The GRU (Gated Recurrent Unit) uses two gates:
- **Update gate** $z_t$: How much of the old hidden state to keep
- **Reset gate** $r_t$: How much of the old state to use when computing the candidate

### Attention as Gating

Whittington et al. (2024) showed that attention-only transformers trained on working memory tasks spontaneously learn **input gating** (via key composition) and **output gating** (via query composition). The self-attention mechanism:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

is a differentiable, content-based routing mechanism. The softmax attention weights determine which inputs are "let through" the gate — analogous to precision-weighting in Active Inference.

In [ ]:
# ── GRU Implementation ────────────────────────────────────────────────────────
class GRU:
    def __init__(self, input_size, hidden_size, output_size):
        scale = 0.1
        self.hidden_size = hidden_size
        # Update gate parameters
        self.Wz = np.random.randn(hidden_size, hidden_size) * scale
        self.Uz = np.random.randn(hidden_size, input_size) * scale
        self.bz = np.zeros(hidden_size)
        # Reset gate parameters
        self.Wr = np.random.randn(hidden_size, hidden_size) * scale
        self.Ur = np.random.randn(hidden_size, input_size) * scale
        self.br = np.zeros(hidden_size)
        # Candidate parameters
        self.Wh = np.random.randn(hidden_size, hidden_size) * scale
        self.Uh = np.random.randn(hidden_size, input_size) * scale
        self.bh = np.zeros(hidden_size)
        # Output
        self.Wy = np.random.randn(output_size, hidden_size) * scale
        self.by = np.zeros(output_size)

    def forward(self, x_seq, h0=None):
        T = len(x_seq)
        if h0 is None:
            h0 = np.zeros(self.hidden_size)
        h = h0
        hiddens, outputs, gate_z, gate_r = [], [], [], []

        for t in range(T):
            x = x_seq[t]
            z = sigmoid(self.Wz @ h + self.Uz @ x + self.bz)  # update gate
            r = sigmoid(self.Wr @ h + self.Ur @ x + self.br)  # reset gate
            h_cand = np.tanh(self.Wh @ (r * h) + self.Uh @ x + self.bh)
            h = (1 - z) * h + z * h_cand  # interpolate
            y = self.Wy @ h + self.by

            hiddens.append(h.copy())
            outputs.append(y.copy())
            gate_z.append(z.copy())
            gate_r.append(r.copy())

        return np.array(hiddens), np.array(outputs), np.array(gate_z), np.array(gate_r)

# Run GRU on flip-flop task
gru = GRU(input_size=3, hidden_size=16, output_size=3)
gru_h, gru_out, gate_z, gate_r = gru.forward(inputs[trial_idx])
print(f"GRU hidden: {gru_h.shape}, gates: z={gate_z.shape}, r={gate_r.shape}")

In [ ]:
# ── Attention Mechanism ───────────────────────────────────────────────────────
def attention(Q, K, V):
    """Scaled dot-product attention.
    Q: (n_queries, d_k)
    K: (n_keys, d_k)
    V: (n_keys, d_v)
    Returns: output (n_queries, d_v), weights (n_queries, n_keys)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    # Softmax over keys for each query
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    return weights @ V, weights

# Demo: Working memory with selective attention
# Scenario: 8 memory slots, query selects which to read
np.random.seed(42)
n_slots = 8
d_model = 4

# Memory contents (keys and values)
K = np.random.randn(n_slots, d_model)
V = np.eye(n_slots, d_model)  # Each slot stores a distinct pattern

# Three different queries: which slot to attend to?
queries = [
    K[2] + np.random.randn(d_model) * 0.1,   # Query similar to slot 2
    K[5] + np.random.randn(d_model) * 0.1,   # Query similar to slot 5
    np.random.randn(d_model) * 0.3,            # Diffuse query
]
query_labels = ['Focused (slot 2)', 'Focused (slot 5)', 'Diffuse (random)']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (q, label) in enumerate(zip(queries, query_labels)):
    Q_mat = q.reshape(1, -1)
    out, w = attention(Q_mat, K, V)
    axes[i].bar(range(n_slots), w[0], color=COLORS['epistemic'], alpha=0.8)
    axes[i].set_xlabel('Memory slot')
    axes[i].set_ylabel('Attention weight')
    axes[i].set_title(f'{label}', fontsize=11)
    axes[i].set_ylim(0, 1)
    axes[i].set_xticks(range(n_slots))

fig.suptitle('Attention as Input Gating: Content-Based Selection',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Focused queries select specific slots; diffuse queries spread attention.")
print("This is precision-weighting: high precision = sharp attention, low precision = diffuse.")

In [ ]:
# ── Visualize GRU Gates ───────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

T = inputs.shape[1]
time = np.arange(T)

# Input pulses
for bit in range(3):
    axes[0].stem(time, inputs[trial_idx, :, bit], linefmt=f'C{bit}-',
                 markerfmt=f'C{bit}o', basefmt='k-', label=f'Bit {bit}')
axes[0].set_ylabel('Input')
axes[0].set_title('GRU Gate Dynamics on Flip-Flop Task', fontsize=14)
axes[0].legend(fontsize=8, loc='upper right')

# Update gate (z) heatmap
im1 = axes[1].imshow(gate_z.T, aspect='auto', cmap='Blues',
                      vmin=0, vmax=1, interpolation='nearest')
axes[1].set_ylabel('Hidden unit')
axes[1].set_title('Update gate $z_t$ (high = write new, low = keep old)', fontsize=11)
plt.colorbar(im1, ax=axes[1], shrink=0.6)

# Reset gate (r) heatmap
im2 = axes[2].imshow(gate_r.T, aspect='auto', cmap='Oranges',
                      vmin=0, vmax=1, interpolation='nearest')
axes[2].set_ylabel('Hidden unit')
axes[2].set_xlabel('Time step')
axes[2].set_title('Reset gate $r_t$ (high = use history, low = ignore history)', fontsize=11)
plt.colorbar(im2, ax=axes[2], shrink=0.6)

plt.tight_layout()
plt.show()

print("Even untrained, the gate structure is visible.")
print("After training, z gates would open briefly at input pulses (write), then close (hold).")

## A3.5 Tiny RNNs + Phase Portraits

Ji-An, Benna & Mattar (2025) made a remarkable discovery: RNNs with just **1 to 4 hidden units**, trained on reward-learning tasks, outperform 30+ classical cognitive models when fit to individual subjects.

Why is this exciting? With so few units, we can **fully visualize** the network's dynamics:
- **2 units** $\Rightarrow$ phase portrait in 2D
- **Fixed points** = stable beliefs or decision states
- **Limit cycles** = oscillating strategies
- **Separatrices** = decision boundaries

This gives us a *dynamical systems* view of cognition: behavior emerges from the geometry of attractors in state space.

In [ ]:
# ── Tiny 2-Unit RNN ──────────────────────────────────────────────────────────
class TinyRNN:
    """A 2-unit RNN for phase portrait analysis."""
    def __init__(self, input_size=1):
        self.hidden_size = 2
        self.Wh = np.array([[1.5, -1.0],
                            [0.8,  1.5]])  # Recurrent weights
        self.Wx = np.random.randn(2, input_size) * 0.5
        self.bh = np.array([0.0, 0.0])

    def step(self, h, x=None):
        if x is None:
            x = np.zeros(self.Wx.shape[1])
        return np.tanh(self.Wh @ h + self.Wx @ x + self.bh)


def plot_phase_portrait(rnn, ax, x_input=None, n_grid=20, title=''):
    """Phase portrait for a 2-unit RNN."""
    h1_range = np.linspace(-1, 1, n_grid)
    h2_range = np.linspace(-1, 1, n_grid)
    H1, H2 = np.meshgrid(h1_range, h2_range)

    dH1, dH2 = np.zeros_like(H1), np.zeros_like(H2)

    for i in range(n_grid):
        for j in range(n_grid):
            h = np.array([H1[i, j], H2[i, j]])
            h_next = rnn.step(h, x_input)
            dH1[i, j] = h_next[0] - h[0]
            dH2[i, j] = h_next[1] - h[1]

    ax.streamplot(h1_range, h2_range, dH1, dH2,
                  color=COLORS['neutral'], density=1.5, linewidth=0.5)
    ax.set_xlabel('$h_1$')
    ax.set_ylabel('$h_2$')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')


def find_fixed_points(rnn, x_input=None, n_init=200, tol=1e-6, max_iter=1000):
    """Find fixed points by iterating from random initial conditions."""
    fixed_points = []
    for _ in range(n_init):
        h = np.random.uniform(-1, 1, 2)
        for _ in range(max_iter):
            h_new = rnn.step(h, x_input)
            if np.linalg.norm(h_new - h) < tol:
                # Check it is not a duplicate
                is_dup = False
                for fp in fixed_points:
                    if np.linalg.norm(fp - h_new) < 1e-4:
                        is_dup = True
                        break
                if not is_dup:
                    fixed_points.append(h_new)
                break
            h = h_new
    return fixed_points

print("TinyRNN and analysis tools defined.")

In [ ]:
# ── Phase Portraits: No Input vs With Input ─────────────────────────────────
np.random.seed(42)
tiny = TinyRNN(input_size=1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# No input
plot_phase_portrait(tiny, axes[0], x_input=None, title='No Input')
fps_none = find_fixed_points(tiny, x_input=None)
for fp in fps_none:
    axes[0].plot(fp[0], fp[1], 'o', color=COLORS['highlight'],
                 markersize=12, markeredgecolor='black', markeredgewidth=1.5, zorder=5)
axes[0].set_title(f'No Input ({len(fps_none)} fixed points)', fontsize=12)

# Positive input
x_pos = np.array([1.0])
plot_phase_portrait(tiny, axes[1], x_input=x_pos, title='Input = +1')
fps_pos = find_fixed_points(tiny, x_input=x_pos)
for fp in fps_pos:
    axes[1].plot(fp[0], fp[1], 'o', color=COLORS['reward'],
                 markersize=12, markeredgecolor='black', markeredgewidth=1.5, zorder=5)
axes[1].set_title(f'Input = +1 ({len(fps_pos)} fixed points)', fontsize=12)

# Negative input
x_neg = np.array([-1.0])
plot_phase_portrait(tiny, axes[2], x_input=x_neg, title='Input = -1')
fps_neg = find_fixed_points(tiny, x_input=x_neg)
for fp in fps_neg:
    axes[2].plot(fp[0], fp[1], 'o', color=COLORS['aif'],
                 markersize=12, markeredgecolor='black', markeredgewidth=1.5, zorder=5)
axes[2].set_title(f'Input = -1 ({len(fps_neg)} fixed points)', fontsize=12)

fig.suptitle('Tiny RNN Phase Portraits: Input Reshapes the Attractor Landscape',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Fixed points (yellow/green/orange dots) = stable states the network converges to.")
print("Input shifts the attractor landscape — like changing beliefs under new evidence.")
print("\nIn AIF terms: fixed points are the modes of the posterior distribution.")
print("Input (observations) shift the posterior — this is Bayesian belief updating!")

In [ ]:
# ── Trajectory Overlay on Phase Portrait ────────────────────────────────────
# Simulate the tiny RNN responding to a sequence of +1/-1 pulses

np.random.seed(7)
T_sim = 80
x_sequence = np.zeros((T_sim, 1))
# Pulses at specific times
x_sequence[10:13] = 1.0
x_sequence[30:33] = -1.0
x_sequence[50:53] = 1.0
x_sequence[65:68] = -1.0

h = np.zeros(2)
traj = [h.copy()]
for t in range(T_sim):
    h = tiny.step(h, x_sequence[t])
    traj.append(h.copy())
traj = np.array(traj)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Phase portrait with trajectory
plot_phase_portrait(tiny, ax1, x_input=None, n_grid=15)
ax1.plot(traj[:, 0], traj[:, 1], '-', color=COLORS['rl'], linewidth=2, alpha=0.8)
ax1.plot(traj[0, 0], traj[0, 1], 'o', color=COLORS['reward'], markersize=10, zorder=5)
ax1.plot(traj[-1, 0], traj[-1, 1], 's', color=COLORS['highlight'], markersize=10, zorder=5)
# Mark pulse times
for t_mark in [10, 30, 50, 65]:
    ax1.plot(traj[t_mark, 0], traj[t_mark, 1], 'D', color=COLORS['aif'],
             markersize=8, zorder=5)
ax1.set_title('Trajectory in Hidden State Space', fontsize=12)

# Time series
time = np.arange(T_sim + 1)
ax2.plot(time, traj[:, 0], color=COLORS['rl'], linewidth=2, label='$h_1$')
ax2.plot(time, traj[:, 1], color=COLORS['aif'], linewidth=2, label='$h_2$')
# Mark input pulses
for t_s, t_e, val in [(10, 13, '+1'), (30, 33, '-1'), (50, 53, '+1'), (65, 68, '-1')]:
    color = COLORS['reward'] if '+' in val else COLORS['epistemic']
    ax2.axvspan(t_s, t_e, alpha=0.2, color=color)
ax2.set_xlabel('Time step')
ax2.set_ylabel('Activation')
ax2.set_title('Hidden State Time Series', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.show()

print("The trajectory jumps between attractors when pulsed — input shifts the landscape,")
print("and the network relaxes to a new fixed point. This is belief updating in action.")

## A3.6 Dynamical Similarity Analysis (DSA) / Koopman

How do we compare the *dynamics* of two different networks? Standard approaches compare representations (e.g., CKA, RSA), but two networks can have very different representations while implementing the same computation.

**Dynamical Similarity Analysis** (Ostrow et al., NeurIPS 2023) addresses this by comparing the *linear operators* that best approximate each network's dynamics. The key tool is **Dynamic Mode Decomposition (DMD)**, a data-driven method for finding the best-fit linear dynamical system:

Given a trajectory $X = [h_1, h_2, \ldots, h_T]$, DMD finds the matrix $A$ such that:
$$h_{t+1} \approx A h_t$$

The eigenvalues of $A$ reveal the dynamical modes:
- $|\lambda| < 1$: Decaying mode (forgetting)
- $|\lambda| = 1$: Sustained mode (memory)
- $|\lambda| > 1$: Growing mode (unstable)
- $\text{Im}(\lambda) \neq 0$: Oscillatory mode

In [ ]:
# ── Dynamic Mode Decomposition ────────────────────────────────────────────────
def dmd(X, r=None):
    """Dynamic Mode Decomposition: find linear operator that best predicts next state.
    
    Args:
        X: trajectory array, shape (T, n_features)
        r: optional rank truncation
    Returns:
        A_dmd: best-fit linear operator
        eigenvalues: eigenvalues of A_dmd
    """
    X1 = X[:-1].T  # (features, T-1)
    X2 = X[1:].T   # (features, T-1)

    U, S, Vt = np.linalg.svd(X1, full_matrices=False)
    if r is not None:
        U, S, Vt = U[:, :r], S[:r], Vt[:r, :]

    # Pseudo-inverse: A = X2 @ X1^+
    A_dmd = X2 @ Vt.T @ np.diag(1.0 / S) @ U.T
    eigenvalues = np.linalg.eigvals(A_dmd)
    return A_dmd, eigenvalues

print("DMD implementation ready.")

In [ ]:
# ── Compare Dynamics: Vanilla RNN vs CTRNN via DMD ──────────────────────────
# Generate trajectories from both architectures with the same input
np.random.seed(42)

n_units_compare = 8
T_compare = 200

# Random input sequence
x_compare = np.zeros((T_compare, n_units_compare))
for t in range(T_compare):
    if np.random.random() < 0.05:
        x_compare[t] = np.random.randn(n_units_compare) * 0.5

# Vanilla RNN trajectory
rnn_compare = VanillaRNN(input_size=n_units_compare, hidden_size=n_units_compare,
                         output_size=n_units_compare)
rnn_traj, _ = rnn_compare.forward(x_compare)

# CTRNN trajectory (run enough steps to match)
ctrnn_compare = CTRNN(n_units_compare, tau=np.ones(n_units_compare) * 5.0, dt=0.5)
# Repeat each input for multiple integration steps
ctrnn_traj = ctrnn_compare.run(np.zeros(n_units_compare), x_seq=x_compare, T=T_compare)
ctrnn_traj = ctrnn_traj[1:]  # remove initial state

# Apply DMD to both
A_rnn, eig_rnn = dmd(rnn_traj, r=n_units_compare)
A_ctrnn, eig_ctrnn = dmd(ctrnn_traj, r=n_units_compare)

# Plot eigenvalue spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Unit circle
theta = np.linspace(0, 2 * np.pi, 100)
for ax in axes:
    ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=0.5, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.3)
    ax.axvline(0, color='k', linewidth=0.3)
    ax.set_aspect('equal')
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')

axes[0].scatter(eig_rnn.real, eig_rnn.imag, c=COLORS['rl'], s=80,
                edgecolors='black', linewidths=0.5, zorder=5)
axes[0].set_title('Vanilla RNN: DMD Eigenvalues', fontsize=12)

axes[1].scatter(eig_ctrnn.real, eig_ctrnn.imag, c=COLORS['aif'], s=80,
                edgecolors='black', linewidths=0.5, zorder=5)
axes[1].set_title('CTRNN ($\\tau$=5): DMD Eigenvalues', fontsize=12)

fig.suptitle('Eigenvalue Spectra on the Unit Circle\n'
             '(inside = decaying, on = sustained, outside = growing)',
             fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print(f"RNN eigenvalue magnitudes: {np.sort(np.abs(eig_rnn))[::-1][:4].round(3)}")
print(f"CTRNN eigenvalue magnitudes: {np.sort(np.abs(eig_ctrnn))[::-1][:4].round(3)}")
print("\nCTRNN eigenvalues tend to cluster closer to the unit circle (slower decay)")
print("because the time constant tau smooths the dynamics.")

In [ ]:
# ── DMD Mode Interpretation ───────────────────────────────────────────────────
# Visualize what each DMD mode represents

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, A, eig, name, color in [
    (axes[0], A_rnn, eig_rnn, 'Vanilla RNN', COLORS['rl']),
    (axes[1], A_ctrnn, eig_ctrnn, 'CTRNN', COLORS['aif'])
]:
    magnitudes = np.abs(eig)
    phases = np.angle(eig) / np.pi  # in units of pi
    sorted_idx = np.argsort(-magnitudes)
    
    bars = ax.bar(range(len(magnitudes)), magnitudes[sorted_idx],
                  color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axhline(1.0, color='k', linestyle='--', linewidth=0.8, alpha=0.5, label='Unit circle')
    ax.set_xlabel('Mode index (sorted by magnitude)')
    ax.set_ylabel('$|\\lambda|$')
    ax.set_title(f'{name}: Mode Magnitudes', fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1.3)

plt.tight_layout()
plt.show()

print("Modes with |lambda| near 1 sustain information (memory).")
print("Modes with |lambda| << 1 decay rapidly (transient processing).")
print("\nDSA compares networks by comparing these spectral signatures,")
print("not the raw hidden representations.")

## A3.7 State Space Models (SSMs)

State Space Models (SSMs) like S4 (Gu et al., 2022) and Mamba (Gu & Dao, 2023) are built on the continuous-time linear system:

$$\dot{x}(t) = Ax(t) + Bu(t), \quad y(t) = Cx(t) + Du(t)$$

This is the same formulation underlying the **Kalman filter** — the optimal Bayesian estimator for linear Gaussian systems. The deep connection:

| SSM Component | Kalman Filter | Active Inference |
|---|---|---|
| $A$ (state transition) | Dynamics model $F$ | Generative model transitions |
| $B$ (input coupling) | Control input $B$ | Action model |
| $C$ (observation) | Observation model $H$ | Likelihood mapping |
| Prediction step | $\hat{x}_{t|t-1} = A\hat{x}_{t-1}$ | Prior belief |
| Update step | $\hat{x}_{t|t} = \hat{x}_{t|t-1} + K(y - C\hat{x}_{t|t-1})$ | Posterior belief (precision-weighted PE) |

The Kalman gain $K$ is literally a **precision-weighting matrix**: it determines how much to trust the observation vs. the prediction. This is Active Inference in its simplest form.

In [ ]:
# ── Kalman Filter: Optimal Bayesian Inference for Linear SSMs ────────────────
class KalmanFilter:
    """1D Kalman filter for tracking a moving target."""
    def __init__(self, A, B, C, Q, R, x0=0.0, P0=1.0):
        self.A = A  # State transition
        self.B = B  # Control input
        self.C = C  # Observation model
        self.Q = Q  # Process noise (state uncertainty)
        self.R = R  # Observation noise (sensory uncertainty)
        self.x = x0  # State estimate
        self.P = P0  # State covariance

    def predict(self, u=0.0):
        """Prediction step: propagate belief forward."""
        self.x = self.A * self.x + self.B * u
        self.P = self.A * self.P * self.A + self.Q
        return self.x, self.P

    def update(self, y):
        """Update step: incorporate observation."""
        # Innovation (prediction error)
        innovation = y - self.C * self.x
        # Innovation covariance
        S = self.C * self.P * self.C + self.R
        # Kalman gain (precision-weighting)
        K = self.P * self.C / S
        # Update state and covariance
        self.x = self.x + K * innovation
        self.P = (1 - K * self.C) * self.P
        return self.x, self.P, K, innovation

# 1D tracking task: a target moves with constant velocity + noise
np.random.seed(42)
T_track = 100
dt_track = 1.0

# True state: position with random walk
true_pos = np.zeros(T_track)
velocity = 0.5
for t in range(1, T_track):
    true_pos[t] = true_pos[t-1] + velocity * dt_track + np.random.randn() * 0.3

# Noisy observations
obs_noise_std = 2.0
observations = true_pos + np.random.randn(T_track) * obs_noise_std

# Run Kalman filter
kf = KalmanFilter(A=1.0, B=dt_track, C=1.0, Q=0.09, R=obs_noise_std**2,
                  x0=0.0, P0=10.0)

estimates = []
uncertainties = []
kalman_gains = []
prediction_errors = []

for t in range(T_track):
    kf.predict(u=velocity)
    x_est, P_est, K, innov = kf.update(observations[t])
    estimates.append(x_est)
    uncertainties.append(np.sqrt(P_est))
    kalman_gains.append(K)
    prediction_errors.append(innov)

estimates = np.array(estimates)
uncertainties = np.array(uncertainties)
kalman_gains = np.array(kalman_gains)
prediction_errors = np.array(prediction_errors)

print(f"Final Kalman gain: {kalman_gains[-1]:.4f}")
print(f"Final uncertainty: {uncertainties[-1]:.4f}")
print(f"Mean absolute error: {np.mean(np.abs(estimates - true_pos)):.3f}")

In [ ]:
# ── Kalman Filter Visualization ───────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

time = np.arange(T_track)

# Tracking
axes[0].plot(time, true_pos, color='black', linewidth=2, label='True position')
axes[0].scatter(time, observations, color=COLORS['neutral'], s=15, alpha=0.4,
                label='Noisy observations', zorder=2)
axes[0].plot(time, estimates, color=COLORS['aif'], linewidth=2, label='Kalman estimate')
axes[0].fill_between(time, estimates - 2*uncertainties, estimates + 2*uncertainties,
                      color=COLORS['aif'], alpha=0.15, label='$\\pm 2\\sigma$ confidence')
axes[0].set_ylabel('Position')
axes[0].set_title('Kalman Filter = Optimal Bayesian Inference for Linear SSMs', fontsize=14)
axes[0].legend(fontsize=9, loc='upper left')

# Kalman gain (precision-weighting)
axes[1].plot(time, kalman_gains, color=COLORS['epistemic'], linewidth=2)
axes[1].set_ylabel('Kalman gain $K$')
axes[1].set_title('Kalman Gain = Precision-Weighting (AIF analog)', fontsize=11)
axes[1].axhline(0.5, linestyle='--', color='gray', alpha=0.5)
axes[1].annotate('High K = trust observations\nLow K = trust prediction',
                 xy=(60, kalman_gains[-1]), fontsize=9, color=COLORS['epistemic'])

# Prediction errors
axes[2].plot(time, prediction_errors, color=COLORS['reward'], linewidth=1, alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('Prediction error')
axes[2].set_xlabel('Time step')
axes[2].set_title('Prediction Errors (should be zero-mean if model is correct)', fontsize=11)

plt.tight_layout()
plt.show()

print("The Kalman filter IS Active Inference for linear Gaussian models:")
print("  - Prediction step = prior belief propagation")
print("  - Update step = precision-weighted prediction error correction")
print("  - Kalman gain K = sensory precision / total precision")

## A3.8 Emerging Synthesis

Looking across these architectures, several convergent themes emerge:

### Multiplicative Couplings (Thalamocortical-Inspired Gating)
The thalamus gates information flow between cortical areas via multiplicative interactions — exactly what LSTM/GRU gates implement. Recent work on **thalamocortical-inspired architectures** makes this connection explicit: attention heads can be seen as thalamic relay nuclei that route information between cortical modules.

### Continuous Thought Machines (Adaptive Compute Time)
Rather than a fixed number of forward-pass steps, some architectures (e.g., Universal Transformers, Continuous Thought Machines) let the network *decide* how long to think. This maps onto the AIF concept of **expected free energy**: the agent continues to process (gather epistemic value) until the expected reduction in uncertainty falls below the cost of computation.

### Symbolic Mechanisms in LLMs
Yang et al. (2025) found evidence of symbolic mechanisms emerging in large language models — compositional, rule-like computations implemented by specific attention head circuits. This resonates with the AIF view that hierarchical generative models should develop compositional structure.

### The Recurring Theme
Across all architectures — CTRNNs, gated RNNs, attention, SSMs — the same computational primitives keep appearing:

| Primitive | Mechanism | AIF Interpretation |
|---|---|---|
| **Gating** | Multiplicative interactions | Precision-weighting |
| **Sustained activity** | Attractors / slow dynamics | Belief maintenance |
| **Context-dependent computation** | Input-dependent flow fields | Policy-conditioned generative model |
| **Temporal hierarchy** | Multiple timescales | Hierarchical predictive coding |

## A3.9 Open Research Directions

1. **Dynamical Interpretability at Scale.** Tiny RNNs (1–4 units) are fully interpretable via phase portraits. Can DSA/DMD-based tools extend this interpretability to networks with hundreds or thousands of units? The challenge is developing dimensionality reduction methods that preserve the *dynamical* structure (fixed points, limit cycles, separatrices) rather than just the representational geometry.

2. **Learned Time Constants as Precision.** CTRNN time constants $\tau_i$ control the effective integration window of each unit. In Active Inference, precision parameters control how much weight is given to different levels of the hierarchy. Can we show a formal equivalence between learned $\tau$ in CTRNNs and learned precision in hierarchical predictive coding? Hasani et al.'s Closed-form Continuous-depth networks (CfCs) are a promising bridge.

3. **Attention as Amortized Inference.** Whittington et al. (2024) showed that attention learns input/output gating. Can we show that multi-head attention *in general* implements amortized variational inference — where each head computes a different component of the approximate posterior? This would unify Transformers with Active Inference at the architectural level.

4. **SSMs + Active Inference.** S4/Mamba's linear recurrence is essentially a discretized Kalman prediction step. Can we build an Active Inference agent whose generative model backbone is an SSM, combining Mamba's efficiency with AIF's principled action selection? The Kalman filter connection (Section A3.7) suggests this is the natural next step.

## A3.10 Exercises

The following exercises progress from guided exploration to open-ended research questions.

In [ ]:
# ── Exercise 1 (Guided): CTRNN Tau and Working Memory Decay ─────────────────
#
# Task: Create a CTRNN with 4 units. Give it a brief input pulse (5 time steps),
# then let it run for 200 time steps with no input. Measure how long the network
# retains a "memory" of the pulse (defined as: norm(h) > 0.1 * norm(h_peak)).
#
# Do this for tau = [1, 2, 5, 10, 20, 50] and plot memory duration vs tau.
#
# Prediction: What relationship do you expect? Linear? Logarithmic?

# YOUR CODE HERE
# Hint:
# taus_to_test = [1, 2, 5, 10, 20, 50]
# memory_durations = []
# for tau_val in taus_to_test:
#     net = CTRNN(4, tau=np.ones(4) * tau_val, dt=0.1)
#     x_pulse = np.zeros((205, 4))
#     x_pulse[:5] = np.random.randn(5, 4)
#     traj = net.run(np.zeros(4), x_seq=x_pulse, T=205)
#     norms = np.linalg.norm(traj, axis=1)
#     peak_norm = np.max(norms[:10])  # peak during/after pulse
#     threshold = 0.1 * peak_norm
#     above = np.where(norms[10:] > threshold)[0]
#     duration = above[-1] if len(above) > 0 else 0
#     memory_durations.append(duration)
#
# plt.figure(figsize=(8, 5))
# plt.plot(taus_to_test, memory_durations, 'o-', color=COLORS['aif'], linewidth=2)
# plt.xlabel('Time constant tau')
# plt.ylabel('Memory duration (time steps)')
# plt.title('Working Memory Duration vs. Time Constant')
# plt.show()

In [ ]:
# ── Exercise 2 (Intermediate): Add GRU Gating to the Tiny RNN ───────────────
#
# Task: Modify TinyRNN to include a GRU-style update gate.
# Then create phase portraits for BOTH the vanilla and gated versions
# (with and without input) and compare.
#
# Questions:
# - How does gating change the location and number of fixed points?
# - Does the gated version have "sharper" basins of attraction?

# YOUR CODE HERE
# Hint:
# class TinyGRU:
#     def __init__(self, input_size=1):
#         self.hidden_size = 2
#         self.Wh = np.array([[1.5, -1.0], [0.8, 1.5]])
#         self.Wx = np.random.randn(2, input_size) * 0.5
#         self.bh = np.zeros(2)
#         # Gate parameters
#         self.Wz_h = np.array([[0.5, 0.0], [0.0, 0.5]])
#         self.Wz_x = np.random.randn(2, input_size) * 0.5
#         self.bz = np.zeros(2)
#
#     def step(self, h, x=None):
#         if x is None:
#             x = np.zeros(self.Wx.shape[1])
#         z = sigmoid(self.Wz_h @ h + self.Wz_x @ x + self.bz)
#         h_cand = np.tanh(self.Wh @ h + self.Wx @ x + self.bh)
#         return (1 - z) * h + z * h_cand
#
# Then use plot_phase_portrait() and find_fixed_points() to compare.

In [ ]:
# ── Exercise 3 (Open-ended): DMD Comparison on Flip-Flop Task ──────────────
#
# Task: Run both the VanillaRNN (16 hidden units) and a CTRNN (16 units)
# on the same flip-flop input sequences. Apply DMD to each trajectory.
# Compare:
#   1. The eigenvalue spectra (which has more sustained modes?)
#   2. The DMD reconstruction error (which is better approximated as linear?)
#   3. The number of "memory modes" (|lambda| > 0.95)
#
# Research question: Does the CTRNN's continuous-time structure
# lead to a more interpretable linear approximation?

# YOUR CODE HERE
# Hint: Reuse the dmd() function from Section A3.6.
# For reconstruction error:
#   X_pred = A_dmd @ X[:-1].T  # predicted next states
#   error = np.mean((X_pred - X[1:].T)**2)
#
# For counting memory modes:
#   n_memory = np.sum(np.abs(eigenvalues) > 0.95)

---

## Further Reading

1. **Hasani, R. et al. (2022).** *Closed-form Continuous-depth Models.* Nature Machine Intelligence, 4, 992–1003. Introduces Closed-form Continuous-depth networks (CfCs) — CTRNNs with analytical solutions, enabling both interpretability and efficiency. The learned time constants are directly inspectable.

2. **Ji-An, Li, Benna, M.K. & Mattar, M.G. (2025).** *Tiny RNNs as cognitive models.* Nature, 637, 122–129. Shows that RNNs with 1–4 hidden units outperform 30+ classical cognitive models on reward-learning tasks, revealing novel behavioral strategies invisible to traditional model fitting.

3. **Ostrow, M. et al. (2023).** *Beyond Geometry: Comparing the Temporal Structure of Computation in Neural Circuits with Dynamical Similarity Analysis.* NeurIPS. Introduces DSA for comparing network dynamics rather than representations. Uses a Koopman/DMD-based approach to extract dynamical structure.

4. **Whittington, J.C.R. et al. (2024).** *Attention as Gating.* Demonstrates that attention-only transformers trained on working memory tasks spontaneously learn input gating (key composition) and output gating (query composition), bridging the gap between attention mechanisms and classical gating theories.

5. **Gu, A., Goel, K. & Ré, C. (2022).** *Efficiently Modeling Long Sequences with Structured State Spaces.* ICLR. Introduces S4, showing that properly parameterized linear SSMs can match or exceed Transformers on long-range sequence tasks.

6. **Beer, R.D. (1995).** *On the Dynamics of Small Continuous-Time Recurrent Neural Networks.* Adaptive Behavior, 3(4), 469–509. The foundational paper on CTRNN dynamics. Exhaustively catalogs the dynamical repertoire (fixed points, limit cycles, chaos) of 1–3 unit CTRNNs.

7. **Gu, A. & Dao, T. (2023).** *Mamba: Linear-Time Sequence Modeling with Selective State Spaces.* Extends S4 with input-dependent (selective) gating, achieving Transformer-level performance with linear compute. The selection mechanism is analogous to precision-weighting in Active Inference.

8. **Yang, Z. et al. (2025).** *Symbolic Mechanisms in Large Language Models.* Evidence for compositional, rule-like computations in specific attention head circuits, suggesting that symbolic reasoning can emerge from subsymbolic architectures.